In [1]:
import sys

In [2]:
print(sys.executable)

c:\Users\divya\Desktop\AI\LangChain_Lab\.venv\Scripts\python.exe


In [4]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from common.llm import get_llm

In [14]:
llm = get_llm(temperature=0)

In [15]:
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from typing import Optional

In [16]:
class LineItem(BaseModel):
    name: str = Field(description="name of the product or service")
    quantity: int = Field(description="Number of units purchased")
    unit_price: float = Field(description="Price per single unit, without symbol")
    

In [17]:
class Invoice(BaseModel):
    invoice_number: str = Field(description="The invoice's unique identification")
    date: str = Field(description="Invoice date in YYYY-MM-DD format")
    billed_to: str = Field(description="Name of the person or company being billed")
    items: list[LineItem] = Field(description="List of all line items in the invoice")
    total: float = Field(description="Toatl Number due")
    payment_terms: Optional[str] = Field(default=None, description="Payment terms, e.g. 'Net 30', if mentioned")

In [20]:
structured_llm = llm.with_structured_output(Invoice)
structured_llm

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.4', 'langchain': '1.4.2'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001AAD9F58180>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001AAD9F58B00>, model_name='openai/gpt-oss-20b', temperature=1e-08, model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Invoice', 'description': '', 'parameters': {'properties': {'invoice_numb

In [21]:
prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are an information extraction assistant. Extract invoice details "
     "from the user's text into the given structure. If a field is not "
     "mentioned in the text, leave it empty or null instead of guessing."),
    ("human", "{text}"),
])

extractor_chain = prompt | structured_llm

In [23]:
sample_invoice = (
    "Invoice #4521, dated March 15 2026. Billed to Rohan Verma. "
    "Items: 2x Wireless Mouse @ $15, 1x Keyboard @ $45. "
    "Total due: $75. Payment terms: Net 30."
)

result = extractor_chain.invoke({"text":sample_invoice})

In [24]:
result

Invoice(invoice_number='4521', date='2026-03-15', billed_to='Rohan Verma', items=[LineItem(name='Wireless Mouse', quantity=2, unit_price=15.0), LineItem(name='Keyboard', quantity=1, unit_price=45.0)], total=75.0, payment_terms='Net 30')

In [26]:
#  Access fields like a normal Python object

print(result.invoice_number)
print(result.date)
print(result.billed_to)
print(result.total)

for item in result.items:
    print(item.name, item.quantity, item.unit_price)

4521
2026-03-15
Rohan Verma
75.0
Wireless Mouse 2 15.0
Keyboard 1 45.0


#### Convert to dict / JSON

In [27]:
result.model_dump()          # Python dict

{'invoice_number': '4521',
 'date': '2026-03-15',
 'billed_to': 'Rohan Verma',
 'items': [{'name': 'Wireless Mouse', 'quantity': 2, 'unit_price': 15.0},
  {'name': 'Keyboard', 'quantity': 1, 'unit_price': 45.0}],
 'total': 75.0,
 'payment_terms': 'Net 30'}

In [29]:
print(result.model_dump_json(indent=2)) # JSON string, nicely formatted

{
  "invoice_number": "4521",
  "date": "2026-03-15",
  "billed_to": "Rohan Verma",
  "items": [
    {
      "name": "Wireless Mouse",
      "quantity": 2,
      "unit_price": 15.0
    },
    {
      "name": "Keyboard",
      "quantity": 1,
      "unit_price": 45.0
    }
  ],
  "total": 75.0,
  "payment_terms": "Net 30"
}


#### Test missing-field handling

In [31]:
sample_no_terms = (
    "Invoice #9981, dated Jan 5 2026. Billed to Meera Nair."
    "Items: 3x Notebook @ $5. Total due: $15."
)

result2 = extractor_chain.invoke({"text": sample_no_terms})
print(result2.payment_terms)

None


In [33]:
print(result2.model_dump_json(indent=2))

{
  "invoice_number": "9981",
  "date": "2026-01-05",
  "billed_to": "Meera Nair",
  "items": [
    {
      "name": "Notebook",
      "quantity": 3,
      "unit_price": 5.0
    }
  ],
  "total": 15.0,
  "payment_terms": null
}


#### .batch() — multiple invoices at once

In [34]:
inputs = [
    {"text": sample_invoice},
    {"text": sample_no_terms},
]

results = extractor_chain.batch(inputs)

for r in results:
    print(r.model_dump_json(indent=2))
    print("---")

{
  "invoice_number": "4521",
  "date": "2026-03-15",
  "billed_to": "Rohan Verma",
  "items": [
    {
      "name": "Wireless Mouse",
      "quantity": 2,
      "unit_price": 15.0
    },
    {
      "name": "Keyboard",
      "quantity": 1,
      "unit_price": 45.0
    }
  ],
  "total": 75.0,
  "payment_terms": "Net 30"
}
---
{
  "invoice_number": "9981",
  "date": "2026-01-05",
  "billed_to": "Meera Nair",
  "items": [
    {
      "name": "Notebook",
      "quantity": 3,
      "unit_price": 5.0
    }
  ],
  "total": 15.0,
  "payment_terms": null
}
---


# Resume Schema

In [35]:
class Resume(BaseModel):
    name: str = Field(description="Name of the Candidate's")
    email: Optional[str] = Field(default=None,description="Email address of the candidates")
    skills: list[str] = Field(description="List of skills mentioned")
    years_experience: Optional[int] = Field(default=None, description="Total years of experience, if stated")

In [40]:
resume_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Your are an information extratcion assistant. Extract resume details"
     "from the user's text into the given structure. If a field is not "
     "mentioned in the text, leave it empty or null istead of guessing"),
    ("human", "{text}"),
])

resume_llm = llm.with_structured_output(Resume)
resume_chain = resume_prompt | resume_llm

In [41]:
sample_resume = "John Doe, john@email.com. Skills: Python, SQL, Docker. 3 years of experience."

result2 = resume_chain.invoke({"text": sample_resume})

In [42]:
print(result2.model_dump_json(indent=2))

{
  "name": "John Doe",
  "email": "john@email.com",
  "skills": [
    "Python",
    "SQL",
    "Docker"
  ],
  "years_experience": 3
}
